# Pipeline — Landing Zone → Bronze (Delta Lake)

Lê os CSVs gravados no bucket `landing-zone` pelo notebook 01 e persiste no bucket `bronze` no formato Delta Lake, com particionamento onde aplicável.

## 1. Configuração

In [ ]:
MINIO_ENDPOINT   = "http://localhost:9010"
MINIO_ACCESS_KEY = "minioadmin"
MINIO_SECRET_KEY = "minioadmin"
LANDING_BUCKET   = "s3a://landing-zone"
BRONZE_BUCKET    = "s3a://bronze"

# Tabelas: pasta no landing-zone -> configurações de escrita no bronze
# partition_by: lista de colunas para particionar (None = sem particionamento)
TABLES = [
    {"name": "funcionarios",   "partition_by": ["status"]},
    {"name": "departamentos",  "partition_by": None},
    {"name": "cargos",         "partition_by": None},
    {"name": "folha_pagamento","partition_by": ["competencia"]},
]

## 2. SparkSession com suporte a S3A e Delta Lake

In [ ]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = (
    SparkSession.builder
    .appName("landing_to_bronze")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.jars.packages",
            "org.apache.hadoop:hadoop-aws:3.3.4,"
            "com.amazonaws:aws-java-sdk-bundle:1.12.262")
    .config("spark.hadoop.fs.s3a.endpoint",               MINIO_ENDPOINT)
    .config("spark.hadoop.fs.s3a.access.key",             MINIO_ACCESS_KEY)
    .config("spark.hadoop.fs.s3a.secret.key",             MINIO_SECRET_KEY)
    .config("spark.hadoop.fs.s3a.path.style.access",      "true")
    .config("spark.hadoop.fs.s3a.impl",                   "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("WARN")

print(f"SparkSession iniciada — versão: {spark.version}")

## 3. Leitura dos CSVs do landing-zone

In [ ]:
from pyspark.sql.functions import current_timestamp, lit

dataframes = {}
for table in TABLES:
    name = table["name"]
    df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(f"{LANDING_BUCKET}/{name}")
        .withColumn("bronze_ingested_at", current_timestamp())
    )
    dataframes[name] = df
    print(f"{name}: {df.count()} registros")

## 4. Gravação em Delta Lake no bucket bronze

In [ ]:
for table in TABLES:
    name         = table["name"]
    partition_by = table["partition_by"]
    df           = dataframes[name]
    output_path  = f"{BRONZE_BUCKET}/{name}"

    writer = df.write.format("delta").mode("overwrite")

    if partition_by:
        writer = writer.partitionBy(*partition_by)

    writer.save(output_path)
    print(f"Delta gravado: {output_path}")

print("\nCarga para a camada bronze concluída.")

## 5. Registro das tabelas no catálogo Spark

In [ ]:
for table in TABLES:
    name        = table["name"]
    delta_path  = f"{BRONZE_BUCKET}/{name}"
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS bronze_{name}
        USING DELTA
        LOCATION '{delta_path}'
    """)
    print(f"Tabela bronze_{name} registrada no catálogo")

print("\nValidação de contagens:")
for table in TABLES:
    name  = table["name"]
    count = spark.sql(f"SELECT COUNT(*) AS n FROM bronze_{name}").collect()[0]["n"]
    print(f"  bronze_{name}: {count} registros")

In [ ]:
spark.stop()
print("SparkSession encerrada.")